# SpaceText — SLM Fine-Tuning on Kaggle T4 GPU

**Phase 2 of the SpaceText BE Major Project**

This notebook fine-tunes **Qwen2.5-3B-Instruct** on a custom aerospace/astrophysics dataset
using 4-bit QLoRA via Unsloth. The trained adapter is then merged and exported as a GGUF
file compatible with Ollama and llama.cpp.

### Dataset
- **27,591 documents** from OpenAlex, NASA News, ISRO, DRDO, arXiv, and ESA
- Preprocessed into instruction-tuning format (Alpaca / ChatML)
- Tasks: taxonomy classification, summarization, concept extraction, domain QA

### Hardware (Kaggle Free Tier)
- GPU: NVIDIA Tesla T4 (16 GB VRAM)
- Runtime: ~3–5 hours for 3 epochs on this dataset

### Steps
1. Install dependencies
2. Mount dataset from Kaggle Datasets
3. Prepare train/val/test splits
4. Fine-tune with QLoRA
5. Merge LoRA → base model
6. Export GGUF for Ollama
7. Download the trained model

## Step 1: Install Dependencies

In [ ]:
# Unsloth — 2x faster fine-tuning, 60% less VRAM
# Must be installed before transformers to avoid conflicts
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q rouge-score bert-score evaluate datasets
print('Dependencies installed ✓')

In [ ]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 2: Locate the Corpus Dataset

In [ ]:
import os
from pathlib import Path

# ─── EDIT THIS PATH ───────────────────────────────────────────────────────────
# After uploading aerospace_corpus.jsonl as a Kaggle Dataset, it will appear at:
# /kaggle/input/<your-dataset-slug>/aerospace_corpus.jsonl
# Replace the slug below with your actual dataset name.
CORPUS_PATH = '/kaggle/input/spacetext-aerospace-corpus/aerospace_corpus.jsonl'
# ──────────────────────────────────────────────────────────────────────────────

if not Path(CORPUS_PATH).exists():
    # Fallback: scan for it automatically
    for root, dirs, files in os.walk('/kaggle/input'):
        for fname in files:
            if fname == 'aerospace_corpus.jsonl':
                CORPUS_PATH = os.path.join(root, fname)
                print(f'Found corpus at: {CORPUS_PATH}')
                break

assert Path(CORPUS_PATH).exists(), f'Corpus not found! Upload aerospace_corpus.jsonl as a Kaggle Dataset first.'

# Count records
with open(CORPUS_PATH) as f:
    n_records = sum(1 for line in f if line.strip())
print(f'Corpus: {n_records:,} records found ✓')

## Step 3: Prepare Dataset (Train / Val / Test Splits)

In [ ]:
import json
import random
from pathlib import Path

OUTPUT_DIR   = Path('/kaggle/working/training_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = (
    'You are SpaceText, an expert aerospace and astrophysics AI assistant. '
    'You classify, summarize, and reason about technical aerospace documents '
    'including satellite systems, propulsion, guidance, structures, and missions.'
)

def format_alpaca(instruction, output, input_ctx=''):
    return {
        'instruction': instruction.strip(),
        'input'      : input_ctx.strip(),
        'output'     : output.strip(),
        'text': (
            f'<|system|>\n{SYSTEM_PROMPT}\n'
            f'<|user|>\n{instruction.strip()}'
            + (f'\n\n{input_ctx.strip()}' if input_ctx.strip() else '')
            + f'\n<|assistant|>\n{output.strip()}'
        ),
    }

examples = []
with open(CORPUS_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        title    = rec.get('title', '').strip()
        abstract = rec.get('abstract', '').strip()
        label    = rec.get('taxonomy_label', 'General Aerospace')
        path     = rec.get('taxonomy_path', [label])
        source   = rec.get('source', '')

        # Built-in pairs from pipeline
        for pair in rec.get('instruction_pairs', []):
            inst = pair.get('instruction', '').strip()
            out  = pair.get('output', '').strip()
            if inst and out:
                examples.append(format_alpaca(inst, out))

        # Extra: concept extraction
        if abstract and len(abstract) > 100:
            examples.append(format_alpaca(
                instruction=(
                    f'Extract the 3 most important technical concepts from the '
                    f'following aerospace document abstract.\n\nTitle: {title}\n\nAbstract: {abstract[:600]}'
                ),
                output=(
                    f'Domain: {label}\n'
                    f'Taxonomy Path: {" > ".join(path)}\n'
                    f'The document discusses: {abstract[:200]}'
                ),
            ))

        # Extra: source-aware classification
        if source and title:
            examples.append(format_alpaca(
                instruction=(
                    f'Given this title from a {source.upper()} technical document, '
                    f'identify the aerospace domain.\n\nTitle: {title}'
                ),
                output=f'Domain: {label}\nTaxonomy Path: {" > ".join(path)}',
            ))

print(f'Total examples: {len(examples):,}')

# Shuffle and split
random.seed(42)
random.shuffle(examples)
n = len(examples)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)
splits  = {
    'train': examples[:n_train],
    'val'  : examples[n_train:n_train + n_val],
    'test' : examples[n_train + n_val:],
}
for name, data in splits.items():
    path_out = OUTPUT_DIR / f'{name}.jsonl'
    with open(path_out, 'w', encoding='utf-8') as f:
        for ex in data:
            f.write(json.dumps(ex, ensure_ascii=False) + '\n')
    print(f'  {name:5s}: {len(data):,} examples → {path_out}')

## Step 4: Load Model with Unsloth (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL  = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LEN = 2048
LORA_RANK   = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,       # auto (bfloat16 on T4)
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    target_modules             = ['q_proj','k_proj','v_proj','o_proj',
                                  'gate_proj','up_proj','down_proj'],
    lora_alpha                 = 32,
    lora_dropout               = 0.05,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state               = 42,
    use_rslora                 = True,
    loftq_config               = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## Step 5: Load Dataset and Train

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import json

def load_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_records = load_jsonl(OUTPUT_DIR / 'train.jsonl')
val_records   = load_jsonl(OUTPUT_DIR / 'val.jsonl')
train_ds = Dataset.from_list([{'text': r['text']} for r in train_records])
val_ds   = Dataset.from_list([{'text': r['text']} for r in val_records])
print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,}')

ADAPTER_OUT = '/kaggle/working/spacetext-adapter'

trainer = SFTTrainer(
    model     = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    args = SFTConfig(
        output_dir                  = ADAPTER_OUT,
        num_train_epochs            = 3,
        per_device_train_batch_size = 2,
        per_device_eval_batch_size  = 2,
        gradient_accumulation_steps = 4,   # effective batch = 8
        warmup_ratio                = 0.03,
        learning_rate               = 2e-4,
        lr_scheduler_type           = 'cosine',
        weight_decay                = 0.01,
        bf16                        = True,
        optim                       = 'adamw_8bit',
        logging_steps               = 50,
        save_steps                  = 500,
        evaluation_strategy         = 'steps',
        eval_steps                  = 500,
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
        report_to                   = 'none',
        seed                        = 42,
        dataset_text_field          = 'text',
        max_seq_length              = MAX_SEQ_LEN,
        packing                     = True,
    ),
)

trainer.train()
trainer.save_model(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print(f'LoRA adapter saved → {ADAPTER_OUT}')

## Step 6: Merge LoRA into Base Model

In [ ]:
MERGED_OUT = '/kaggle/working/spacetext-merged'

model.save_pretrained_merged(
    MERGED_OUT,
    tokenizer,
    save_method='merged_16bit',
)
print(f'Merged model saved → {MERGED_OUT}')

## Step 7: Export GGUF for Ollama / llama.cpp

In [ ]:
GGUF_OUT = '/kaggle/working/spacetext-gguf'

model.save_pretrained_gguf(
    GGUF_OUT,
    tokenizer,
    quantization_method='q4_k_m',  # ~2GB, best quality/size for 3B
)
print(f'GGUF model saved → {GGUF_OUT}')

# List output files for download
import os
for root, dirs, files in os.walk('/kaggle/working'):
    for fname in files:
        full = os.path.join(root, fname)
        size_mb = os.path.getsize(full) / 1e6
        print(f'  {full}  ({size_mb:.1f} MB)')

## Step 8: Quick Inference Test
Verify the model can answer aerospace questions before downloading.

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)  # switch to inference mode

SYSTEM = (
    'You are SpaceText, an expert aerospace and astrophysics AI assistant. '
    'You classify, summarize, and reason about technical aerospace documents.'
)

test_questions = [
    'What propulsion system does the GSLV Mk III use for its cryogenic upper stage?',
    'Categorize this document: "Analysis of hall-effect thrusters for small satellite orbital transfer missions"',
    'Summarize the key contribution of this paper: Ion thruster grid erosion modelling using molecular dynamics simulation.',
]

for q in test_questions:
    prompt = f'<|system|>\n{SYSTEM}\n<|user|>\n{q}\n<|assistant|>\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with __import__('torch').no_grad():
        out = model.generate(**inputs, max_new_tokens=150, do_sample=False, repetition_penalty=1.1)
    answer = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'Q: {q}')
    print(f'A: {answer}')
    print('-' * 60)

## Done! Download Your Model

Go to the **Output** tab on the right side of Kaggle and download:
- `spacetext-adapter/` — LoRA adapter only (~180 MB, use with base model)
- `spacetext-merged/` — Full merged model in fp16 (~6 GB)
- `spacetext-gguf/spacetext-q4_k_m.gguf` — **Recommended for local use** (~2 GB)

To run the GGUF locally with Ollama:
```bash
ollama create spacetext -f Modelfile
ollama run spacetext
```